In [ ]:
#@title Cell 1 - Notebook overview

from IPython.display import display, Markdown

display(Markdown(r"""
# Simulation 05: Chromosome–plasmid association

Simulation 5 keeps the finalized Simulation 1 biological model unchanged and changes only the pattern of observed pathogen–plasmid combinations.

The model remains

\[
y_{ij}=\alpha+\boldsymbol{\beta}_C^T\mathbf c_i+\boldsymbol{\beta}_P^T\mathbf p_j+\mathbf c_i^T\mathbf B\mathbf p_j+u_i+\varepsilon_{ij}.
\]

Two designs are compared with the **same number of observations**:

- **weak association:** plasmids are broadly mixed across chromosomal backgrounds;
- **strong association:** plasmids are preferentially observed in restricted chromosomal backgrounds.

For every pathogen, \(P_0\) is observed and 14 of 20 plasmid-containing states are observed. Thus each design has 3,000 observations, as in Simulation 2.

The true complete-grid \(\Delta_{ij}\) and \(\Delta\Delta_{ik,j}\) remain known. The primary test is whether stronger chromosome–plasmid association worsens recovery while observation count is held constant.
"""))

print("Transition: Cell 2 loads the empirical plasmid features and defines the fixed Simulation 5 settings.")

print(
    "Redesigned Simulation 5: full rank is guaranteed by only a minimal "
    "61-pathogen x 5-plasmid anchor cross; association is free to vary in "
    "the remaining observation slots."
)


In [ ]:
#@title Cell 2 - Load public plasmid-feature input and define fixed settings

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd

from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize_scalar

REPO_ROOT = Path.cwd()
DATA_FILE = REPO_ROOT / "data" / "plasmid_feature_pairs.csv"

OUTPUT_DIR = (
    REPO_ROOT
    / "results"
    / "simulation_05_chromosome_plasmid_association"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Public plasmid-feature input was not found:\n"
        f"{DATA_FILE}\n"
        "Run the notebook from the repository root."
    )

empirical_pairs = pd.read_csv(
    DATA_FILE,
    low_memory=False,
)

KEY_SITE_COLUMNS = [
    "sutcliffe_32_nt",
    "sutcliffe_162_nt",
    "sutcliffe_175_nt",
]

required_columns = [
    *KEY_SITE_COLUMNS,
    "CN_TEM1",
]

missing_columns = [
    c for c in required_columns
    if c not in empirical_pairs.columns
]

if missing_columns:
    raise RuntimeError(
        "The public plasmid-feature input is missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

empirical_pairs = empirical_pairs[
    required_columns
].copy()

for col in KEY_SITE_COLUMNS:
    empirical_pairs[col] = (
        empirical_pairs[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

empirical_pairs["CN_TEM1"] = pd.to_numeric(
    empirical_pairs["CN_TEM1"],
    errors="coerce",
)

valid_nt = {"A", "C", "G", "T"}

complete_mask = (
    empirical_pairs[KEY_SITE_COLUMNS]
    .apply(lambda s: s.isin(valid_nt))
    .all(axis=1)
    & empirical_pairs["CN_TEM1"].notna()
    & (empirical_pairs["CN_TEM1"] > 0)
)

empirical_pairs = (
    empirical_pairs.loc[complete_mask]
    .reset_index(drop=True)
)

if len(empirical_pairs) < 20:
    raise RuntimeError(
        f"Only {len(empirical_pairs)} complete promoter/copy-number pairs remain; "
        "at least 20 are required."
    )

CN_REFERENCE = float(
    empirical_pairs["CN_TEM1"].median()
)

if not np.isfinite(CN_REFERENCE) or CN_REFERENCE <= 0:
    raise RuntimeError(
        "The median CN_TEM1 is not positive."
    )

# -------------------------------------------------------------------------
# Simulation 1 biological settings: unchanged
# -------------------------------------------------------------------------

MASTER_SEED = 20260906

N_PATHOGENS = 200
N_PLASMIDS = 20

PREDEFINED_GENES = [
    "acrB", "acrR", "ampC", "basR", "cirA", "cyaA", "fabI", "folP", "ftsI",
    "gyrA", "marR", "nfsA", "nfsB", "ompC", "ompF", "parC", "parE", "pmrB",
    "ptsI", "rpoB", "rpsL", "soxR", "soxS", "uhpT", "acrA", "tolC", "marA",
    "rob", "ompR", "envZ",
]

TARGET_FEATURE_LABELS = []
for gene in PREDEFINED_GENES:
    TARGET_FEATURE_LABELS.extend([
        f"{gene}:coding",
        f"{gene}:upstream_300bp",
    ])

D_C = len(TARGET_FEATURE_LABELS)

PLASMID_FEATURE_LABELS = [
    "TEM1_plasmid_presence",
    "C32T",
    "G162T",
    "G175A",
    "log2_CN_relative_to_empirical_median",
]
D_P = len(PLASMID_FEATURE_LABELS)

CHROMOSOMAL_STATES = np.array([-1.0, 0.0, 1.0])
CHROMOSOMAL_STATE_PROBS = np.array([0.15, 0.70, 0.15])

N_BACKGROUND_SNPS = 500
ALLELE_FREQ_LOW = 0.10
ALLELE_FREQ_HIGH = 0.90

ALPHA_TRUE = -2.5

SIGMA_G_TRUE = 0.40
SIGMA_E_TRUE = 0.32
SIGMA_G2_TRUE = SIGMA_G_TRUE ** 2
SIGMA_E2_TRUE = SIGMA_E_TRUE ** 2

BETA_P_TRUE = np.array([
    0.50,
    0.50,
    0.25,
    0.00,
    0.25,
], dtype=float)

INTERACTION_MAGNITUDE = 0.10

# -------------------------------------------------------------------------
# Simulation 5 observation-pattern settings
# -------------------------------------------------------------------------

N_OBSERVED_PLASMIDS_PER_PATHOGEN = 14
WEAK_ASSOCIATION_STRENGTH = 0.0
STRONG_ASSOCIATION_STRENGTH = 4.0
MIN_PATHOGENS_PER_PLASMID = 40

N_SIM_REPLICATES = 100
N_BOOTSTRAP = 200
RUN_FULL_BOOTSTRAP_COVERAGE = False

print("=" * 90)
print("SIMULATION 05 — FIXED SETTINGS")
print("=" * 90)
print(f"Empirical promoter/CN pairs available: {len(empirical_pairs):,}")
print(f"Empirical median CN_TEM1:               {CN_REFERENCE:.6f}")
print(f"Pathogens:                              {N_PATHOGENS}")
print(f"Plasmids:                               {N_PLASMIDS}")
print(f"Observed plasmids per pathogen:         {N_OBSERVED_PLASMIDS_PER_PATHOGEN}")
print(f"Observed P+ combinations per design:    {N_PATHOGENS * N_OBSERVED_PLASMIDS_PER_PATHOGEN:,}")
print(f"Total observations per design:          {N_PATHOGENS * (N_OBSERVED_PLASMIDS_PER_PATHOGEN + 1):,}")
print(f"Weak association strength:              {WEAK_ASSOCIATION_STRENGTH}")
print(f"Strong association strength:            {STRONG_ASSOCIATION_STRENGTH}")
print(f"alpha:                                  {ALPHA_TRUE}")
print(f"sigma_g:                                {SIGMA_G_TRUE}")
print(f"sigma_e:                                {SIGMA_E_TRUE}")
print(f"Output directory:                       {OUTPUT_DIR}")
print("\\nCell 2: PASS")


In [ ]:
#@title Cell 3 - Define Simulation 1 biology and chromosome–plasmid association masks

def feature_index(gene, region):
    label = (
        f"{gene}:coding"
        if region == "coding"
        else f"{gene}:upstream_300bp"
    )
    return TARGET_FEATURE_LABELS.index(label)


def build_true_chromosomal_coefficients():
    beta_C = np.zeros(D_C, dtype=float)
    B = np.zeros((D_C, D_P), dtype=float)

    efflux_machinery = {"acrA", "acrB", "tolC"}
    repressors = {"acrR", "marR"}
    activators = {"marA", "rob", "soxR", "soxS"}
    porins = {"ompC", "ompF"}

    for gene in PREDEFINED_GENES:
        coding_i = feature_index(gene, "coding")
        upstream_i = feature_index(gene, "upstream")

        if gene in efflux_machinery:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in repressors:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene in activators:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in porins:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene == "ampC":
            beta_C[coding_i] = +0.10
            beta_C[upstream_i] = +0.25

        elif gene == "ftsI":
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.10

        if gene in efflux_machinery:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in repressors:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

        elif gene in activators:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in porins:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

    return beta_C, B


BETA_C_TRUE, B_TRUE = build_true_chromosomal_coefficients()


def simulate_targeted_chromosomal_features(rng):
    for _ in range(100):
        C = rng.choice(
            CHROMOSOMAL_STATES,
            size=(N_PATHOGENS, D_C),
            p=CHROMOSOMAL_STATE_PROBS,
        ).astype(float)

        augmented = np.column_stack([
            np.ones(N_PATHOGENS, dtype=float),
            C,
        ])

        if np.linalg.matrix_rank(augmented) == D_C + 1:
            return C

    raise RuntimeError(
        "Could not generate a full-rank pathogen chromosome matrix."
    )


def sample_empirical_plasmids(rng):
    n_empirical = len(empirical_pairs)

    for _ in range(5000):
        selected_positions = rng.choice(
            n_empirical,
            size=N_PLASMIDS,
            replace=False,
        )

        sampled = (
            empirical_pairs.iloc[selected_positions]
            .copy()
            .reset_index(drop=True)
        )

        sampled.insert(
            0,
            "plasmid_id",
            [f"P{j}" for j in range(1, N_PLASMIDS + 1)],
        )

        sampled["I_C32T"] = (
            sampled["sutcliffe_32_nt"].eq("T")
        ).astype(float)

        sampled["I_G162T"] = (
            sampled["sutcliffe_162_nt"].eq("T")
        ).astype(float)

        sampled["I_G175A"] = (
            sampled["sutcliffe_175_nt"].eq("A")
        ).astype(float)

        sampled["q_CN"] = np.log2(
            sampled["CN_TEM1"].astype(float)
            / CN_REFERENCE
        )

        P = np.column_stack([
            np.ones(N_PLASMIDS, dtype=float),
            sampled["I_C32T"].to_numpy(dtype=float),
            sampled["I_G162T"].to_numpy(dtype=float),
            sampled["I_G175A"].to_numpy(dtype=float),
            sampled["q_CN"].to_numpy(dtype=float),
        ])

        P_all = np.vstack([
            np.zeros((1, D_P), dtype=float),
            P,
        ])

        augmented_state_matrix = np.column_stack([
            np.ones(N_PLASMIDS + 1, dtype=float),
            P_all,
        ])

        if np.linalg.matrix_rank(augmented_state_matrix) == D_P + 1:
            return P, sampled

    raise RuntimeError(
        "Could not sample 20 full-rank empirical plasmid profiles."
    )


def simulate_background_relatedness(rng):
    source_frequencies = rng.uniform(
        ALLELE_FREQ_LOW,
        ALLELE_FREQ_HIGH,
        size=N_BACKGROUND_SNPS,
    )

    G = rng.binomial(
        1,
        source_frequencies,
        size=(N_PATHOGENS, N_BACKGROUND_SNPS),
    ).astype(float)

    p = G.mean(axis=0)
    Z = G - p[None, :]

    denominator = float(
        np.sum(p * (1.0 - p))
    )

    if denominator <= 0:
        raise ValueError(
            "Background-SNP relatedness denominator is not positive."
        )

    K = (Z @ Z.T) / denominator
    K = (K + K.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(K)

    if eigenvalues.min() < -1e-8:
        raise ValueError(
            f"Constructed K is unexpectedly non-PSD: "
            f"minimum eigenvalue={eigenvalues.min():.6g}"
        )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    K = (
        eigenvectors * eigenvalues
    ) @ eigenvectors.T

    K = (K + K.T) / 2.0

    return K, G, p


def build_complete_design(C, P):
    P_all = np.vstack([
        np.zeros((1, D_P), dtype=float),
        P,
    ])

    n_states = N_PLASMIDS + 1

    pathogen_index = np.repeat(
        np.arange(N_PATHOGENS, dtype=int),
        n_states,
    )

    plasmid_state_index = np.tile(
        np.arange(n_states, dtype=int),
        N_PATHOGENS,
    )

    C_obs = C[pathogen_index, :]
    P_obs = P_all[plasmid_state_index, :]

    interaction = np.einsum(
        "ni,nj->nij",
        C_obs,
        P_obs,
    ).reshape(
        len(pathogen_index),
        D_C * D_P,
    )

    X = np.column_stack([
        np.ones(len(pathogen_index), dtype=float),
        C_obs,
        P_obs,
        interaction,
    ])

    return (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    )


def draw_correlated_host_effect(rng, K, sigma_g2):
    eigenvalues, eigenvectors = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    z = rng.normal(
        0.0,
        1.0,
        size=N_PATHOGENS,
    )

    u = eigenvectors @ (
        np.sqrt(
            sigma_g2 * eigenvalues
        ) * z
    )

    return u


def simulate_complete_dataset(seed):
    """
    Generate exactly the same complete biological dataset structure as Simulation 1.
    Missingness is applied only afterwards.
    """
    rng = np.random.default_rng(seed)

    C = simulate_targeted_chromosomal_features(rng)
    P, sampled_plasmids = sample_empirical_plasmids(rng)

    (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    ) = build_complete_design(C, P)

    K, G_background, background_frequencies = (
        simulate_background_relatedness(rng)
    )

    theta_true = np.concatenate([
        np.array([ALPHA_TRUE], dtype=float),
        BETA_C_TRUE,
        BETA_P_TRUE,
        B_TRUE.reshape(-1),
    ])

    structural_mean = X @ theta_true

    u = draw_correlated_host_effect(
        rng,
        K,
        SIGMA_G2_TRUE,
    )

    epsilon = rng.normal(
        0.0,
        SIGMA_E_TRUE,
        size=X.shape[0],
    )

    y = (
        structural_mean
        + u[pathogen_index]
        + epsilon
    )

    return {
        "seed": int(seed),
        "C": C,
        "P": P,
        "P_all": P_all,
        "sampled_plasmids": sampled_plasmids,
        "K": K,
        "G_background": G_background,
        "background_frequencies": background_frequencies,
        "beta_C_true": BETA_C_TRUE.copy(),
        "beta_P_true": BETA_P_TRUE.copy(),
        "B_true": B_TRUE.copy(),
        "theta_true": theta_true,
        "u_true": u,
        "epsilon_true": epsilon,
        "structural_mean": structural_mean,
        "X_complete": X,
        "pathogen_index_complete": pathogen_index,
        "plasmid_state_index_complete": plasmid_state_index,
        "y_complete": y,
    }


def chromosome_association_axis(C):
    """
    Use chromosome PC1 only to construct the observation pattern.
    It does not enter the MIC-generating model as a new effect.
    """
    centered = C - C.mean(
        axis=0,
        keepdims=True,
    )

    U, S, Vt = np.linalg.svd(
        centered,
        full_matrices=False,
    )

    score = U[:, 0] * S[0]

    score = (
        score - score.mean()
    ) / score.std(ddof=1)

    if score[
        np.argmax(
            np.abs(score)
        )
    ] < 0:
        score = -score

    return score


def select_minimal_full_rank_anchor_sets(C, P):
    """
    Select the smallest complete factorial anchor sets needed to guarantee
    full rank of the 366-column fixed-effect model.

    We need:
      - 61 pathogen rows with full-rank [1, C] representation;
      - 5 plasmid rows with full-rank P representation;
      - P0, which is already observed for every pathogen.

    The resulting anchor factorial contains only
        61 x (1 + 5) = 366 observations,
    exactly enough to guarantee rank 366.

    Unlike the previous notebook, the five anchor plasmids are NOT observed
    in all 200 pathogens. They are forced only in the 61 anchor pathogens.
    """
    from scipy.linalg import qr

    C_augmented = np.column_stack([
        np.ones(
            N_PATHOGENS,
            dtype=float,
        ),
        C,
    ])

    _, _, pathogen_pivots = qr(
        C_augmented.T,
        pivoting=True,
        mode="economic",
    )

    anchor_pathogens = np.sort(
        np.asarray(
            pathogen_pivots[
                : D_C + 1
            ],
            dtype=int,
        )
    )

    if np.linalg.matrix_rank(
        C_augmented[
            anchor_pathogens,
            :
        ]
    ) != D_C + 1:
        raise RuntimeError(
            "Could not select 61 linearly independent pathogen backgrounds."
        )

    _, _, plasmid_pivots = qr(
        P.T,
        pivoting=True,
        mode="economic",
    )

    anchor_plasmids = np.sort(
        np.asarray(
            plasmid_pivots[
                :D_P
            ],
            dtype=int,
        )
    )

    if np.linalg.matrix_rank(
        P[
            anchor_plasmids,
            :
        ]
    ) != D_P:
        raise RuntimeError(
            "Could not select five linearly independent plasmids."
        )

    return (
        anchor_pathogens,
        anchor_plasmids,
    )


def build_minimal_anchor_subdesign(
    C,
    P,
    anchor_pathogens,
    anchor_plasmids,
):
    """
    Build the 61-pathogen x (P0 + 5 plasmids) anchor factorial used only
    for one-time QC. Rank 366 guarantees identifiability of the full model.
    """
    C_anchor = C[
        anchor_pathogens,
        :
    ]

    P_states = np.vstack([
        np.zeros(
            (
                1,
                D_P,
            ),
            dtype=float,
        ),
        P[
            anchor_plasmids,
            :
        ],
    ])

    n_anchor_pathogens = len(
        anchor_pathogens
    )

    n_states = (
        1
        + len(
            anchor_plasmids
        )
    )

    pathogen_local_index = np.repeat(
        np.arange(
            n_anchor_pathogens,
            dtype=int,
        ),
        n_states,
    )

    state_index = np.tile(
        np.arange(
            n_states,
            dtype=int,
        ),
        n_anchor_pathogens,
    )

    C_obs = C_anchor[
        pathogen_local_index,
        :
    ]

    P_obs = P_states[
        state_index,
        :
    ]

    interaction = np.einsum(
        "ni,nj->nij",
        C_obs,
        P_obs,
    ).reshape(
        len(
            pathogen_local_index
        ),
        D_C * D_P,
    )

    X_anchor = np.column_stack([
        np.ones(
            len(
                pathogen_local_index
            ),
            dtype=float,
        ),
        C_obs,
        P_obs,
        interaction,
    ])

    return X_anchor


def repair_minimum_plasmid_support(
    observed_pplus_grid,
    mandatory_grid,
    priority_matrix,
    min_count,
):
    """
    Ensure every plasmid is represented in at least min_count pathogens.

    Repair uses within-pathogen swaps and never removes a mandatory anchor
    observation. Therefore:
      - every pathogen keeps exactly 14 observed P+ states;
      - the minimal full-rank anchor factorial remains intact;
      - association is altered only as much as necessary.
    """
    observed = observed_pplus_grid.copy()

    counts = observed.sum(
        axis=0
    ).astype(int)

    max_swaps = (
        N_PATHOGENS
        * N_PLASMIDS
    )

    swaps_done = 0

    for _ in range(
        max_swaps
    ):
        under = np.where(
            counts < min_count
        )[0]

        if len(
            under
        ) == 0:
            return (
                observed,
                swaps_done,
            )

        j_add = int(
            under[
                np.argmin(
                    counts[
                        under
                    ]
                )
            ]
        )

        best = None

        candidate_hosts = np.where(
            ~observed[
                :,
                j_add,
            ]
        )[0]

        for i in candidate_hosts:
            removable = np.where(
                observed[
                    i,
                    :
                ]
                & (
                    ~mandatory_grid[
                        i,
                        :
                    ]
                )
                & (
                    counts
                    > min_count
                )
            )[0]

            if len(
                removable
            ) == 0:
                continue

            losses = (
                priority_matrix[
                    i,
                    removable,
                ]
                - priority_matrix[
                    i,
                    j_add,
                ]
            )

            local = int(
                np.argmin(
                    losses
                )
            )

            j_remove = int(
                removable[
                    local
                ]
            )

            loss = float(
                losses[
                    local
                ]
            )

            if (
                best is None
                or loss < best[0]
            ):
                best = (
                    loss,
                    int(i),
                    j_remove,
                )

        if best is None:
            raise RuntimeError(
                "Could not repair minimum plasmid support while preserving "
                "the minimal full-rank anchor factorial."
            )

        _, i_swap, j_remove = best

        observed[
            i_swap,
            j_remove,
        ] = False

        observed[
            i_swap,
            j_add,
        ] = True

        counts[
            j_remove
        ] -= 1

        counts[
            j_add
        ] += 1

        swaps_done += 1

    raise RuntimeError(
        "Minimum-support repair exceeded the maximum allowed swaps."
    )


def generate_association_mask(
    C,
    P,
    association_strength,
    seed,
):
    """
    Generate weak or strong chromosome-plasmid association.

    A minimal 61-pathogen x 5-plasmid anchor cross guarantees full rank but
    occupies only 305 of the 2,800 observed P+ combinations.

    All remaining observation slots follow the association rule:
      - strength 0: random mixing;
      - larger strength: plasmids closer to the pathogen chromosome PC1 score
        are preferentially observed.

    This avoids the previous universal-anchor design, which made the strong
    association condition too well mixed and removed the intended stress test.
    """
    chromosome_score = chromosome_association_axis(
        C
    )

    plasmid_centres = np.quantile(
        chromosome_score,
        np.linspace(
            0.025,
            0.975,
            N_PLASMIDS,
        ),
    )

    (
        anchor_pathogens,
        anchor_plasmids,
    ) = select_minimal_full_rank_anchor_sets(
        C,
        P,
    )

    mandatory_grid = np.zeros(
        (
            N_PATHOGENS,
            N_PLASMIDS,
        ),
        dtype=bool,
    )

    mandatory_grid[
        np.ix_(
            anchor_pathogens,
            anchor_plasmids,
        )
    ] = True

    rng = np.random.default_rng(
        seed
    )

    distance = np.abs(
        chromosome_score[
            :,
            None,
        ]
        - plasmid_centres[
            None,
            :
        ]
    )

    priority_matrix = (
        -association_strength
        * distance
        + rng.gumbel(
            size=(
                N_PATHOGENS,
                N_PLASMIDS,
            )
        )
    )

    observed_pplus_grid = (
        mandatory_grid.copy()
    )

    for i in range(
        N_PATHOGENS
    ):
        already_observed = int(
            observed_pplus_grid[
                i,
                :
            ].sum()
        )

        n_needed = (
            N_OBSERVED_PLASMIDS_PER_PATHOGEN
            - already_observed
        )

        if n_needed < 0:
            raise RuntimeError(
                "Mandatory anchor observations exceed the allowed "
                "P+ observations per pathogen."
            )

        if n_needed == 0:
            continue

        candidates = np.where(
            ~observed_pplus_grid[
                i,
                :
            ]
        )[0]

        candidate_priority = (
            priority_matrix[
                i,
                candidates,
            ]
        )

        chosen_local = np.argpartition(
            candidate_priority,
            -n_needed,
        )[
            -n_needed:
        ]

        chosen = candidates[
            chosen_local
        ]

        observed_pplus_grid[
            i,
            chosen,
        ] = True

    (
        observed_pplus_grid,
        support_repair_swaps,
    ) = repair_minimum_plasmid_support(
        observed_pplus_grid,
        mandatory_grid,
        priority_matrix,
        MIN_PATHOGENS_PER_PLASMID,
    )

    per_pathogen_counts = (
        observed_pplus_grid.sum(
            axis=1
        )
    )

    if not np.all(
        per_pathogen_counts
        == N_OBSERVED_PLASMIDS_PER_PATHOGEN
    ):
        raise RuntimeError(
            "Association-mask construction changed the required number "
            "of observed plasmids per pathogen."
        )

    plasmid_counts = (
        observed_pplus_grid.sum(
            axis=0
        )
    )

    if (
        plasmid_counts.min()
        < MIN_PATHOGENS_PER_PLASMID
    ):
        raise RuntimeError(
            "Association-mask construction failed the minimum-support rule."
        )

    grid21 = np.zeros(
        (
            N_PATHOGENS,
            N_PLASMIDS + 1,
        ),
        dtype=bool,
    )

    grid21[
        :,
        0,
    ] = True

    grid21[
        :,
        1:
    ] = observed_pplus_grid

    observed_mask = (
        grid21.ravel()
    )

    oi, oj = np.where(
        observed_pplus_grid
    )

    obs_score = (
        chromosome_score[
            oi
        ]
    )

    obs_centre = (
        plasmid_centres[
            oj
        ]
    )

    corr = np.corrcoef(
        obs_score,
        obs_centre,
    )[0, 1]

    mean_distance = float(
        np.mean(
            np.abs(
                obs_score
                - obs_centre
            )
        )
    )

    return {
        "observed_mask":
            observed_mask,

        "observed_pplus_grid":
            observed_pplus_grid,

        "withheld_pplus_grid":
            ~observed_pplus_grid,

        "chromosome_score":
            chromosome_score,

        "plasmid_centres":
            plasmid_centres,

        "plasmid_counts":
            plasmid_counts,

        "score_centre_correlation":
            float(
                corr
            ),

        "mean_abs_score_centre_distance":
            mean_distance,

        "association_strength":
            float(
                association_strength
            ),

        "anchor_pathogen_indices":
            anchor_pathogens.copy(),

        "anchor_plasmid_indices":
            anchor_plasmids.copy(),

        "anchor_pathogen_ids": [
            f"C{i + 1}"
            for i in anchor_pathogens
        ],

        "anchor_plasmid_ids": [
            f"P{j + 1}"
            for j in anchor_plasmids
        ],

        "mandatory_anchor_Pplus_count":
            int(
                mandatory_grid.sum()
            ),

        "support_repair_swaps":
            int(
                support_repair_swaps
            ),
    }


def prepare_association_dataset(seed):
    dataset = simulate_complete_dataset(seed)

    weak = generate_association_mask(
        dataset["C"], dataset["P"],
        WEAK_ASSOCIATION_STRENGTH, seed + 10_000_000,
    )
    strong = generate_association_mask(
        dataset["C"], dataset["P"],
        STRONG_ASSOCIATION_STRENGTH, seed + 20_000_000,
    )

    dataset["weak"] = weak
    dataset["strong"] = strong

    for label in ["weak", "strong"]:
        mask = dataset[label]["observed_mask"]
        dataset[label]["X"] = dataset["X_complete"][mask, :]
        dataset[label]["y"] = dataset["y_complete"][mask]
        dataset[label]["pathogen_index"] = dataset["pathogen_index_complete"][mask]
        dataset[label]["plasmid_state_index"] = dataset["plasmid_state_index_complete"][mask]

    return dataset


print("Cell 3: PASS")


In [ ]:
#@title Cell 4 - Define efficient REML and GLS fitting

def prepare_reml_static(X, pathogen_index, K):
    X = np.asarray(X, dtype=float)
    K = np.asarray(K, dtype=float)

    m, p_fixed = X.shape

    design_rank = np.linalg.matrix_rank(X)

    if design_rank != p_fixed:
        raise ValueError(
            f"Fixed-effect design matrix is rank deficient: "
            f"rank={design_rank}, columns={p_fixed}."
        )

    eigenvalues, Q = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    keep = eigenvalues > 1e-10
    eigenvalues = eigenvalues[keep]
    Q = Q[:, keep]

    B_lowrank = (
        Q[pathogen_index, :]
        * np.sqrt(eigenvalues)[None, :]
    )

    static = {
        "m": int(m),
        "p_fixed": int(p_fixed),
        "rank_K": int(len(eigenvalues)),
        "B_lowrank": B_lowrank,
        "BtB": B_lowrank.T @ B_lowrank,
        "BtX": B_lowrank.T @ X,
        "XTX": X.T @ X,
        "X": X,
    }

    return static


def add_y_to_reml_static(static, y):
    working = {
        key: value
        for key, value in static.items()
        if key not in {"B_lowrank", "X"}
    }

    B_lowrank = static["B_lowrank"]
    X = static["X"]

    working["Bty"] = B_lowrank.T @ y
    working["Xty"] = X.T @ y
    working["yty"] = float(y @ y)

    return working


def evaluate_profile_reml(log_delta, working, return_fit=False):
    delta = float(np.exp(log_delta))

    m = working["m"]
    p_fixed = working["p_fixed"]
    rank_K = working["rank_K"]

    M = (
        np.eye(rank_K)
        + working["BtB"] / delta
    )

    try:
        chol_M = cho_factor(
            M,
            lower=True,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    M_inv_BtX = cho_solve(
        chol_M,
        working["BtX"],
        check_finite=False,
    )

    M_inv_Bty = cho_solve(
        chol_M,
        working["Bty"],
        check_finite=False,
    )

    XtAinvX = (
        working["XTX"] / delta
        - (
            working["BtX"].T
            @ M_inv_BtX
        ) / (delta ** 2)
    )

    XtAinvy = (
        working["Xty"] / delta
        - (
            working["BtX"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    yAinvy = (
        working["yty"] / delta
        - float(
            working["Bty"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    XtAinvX = (
        XtAinvX + XtAinvX.T
    ) / 2.0

    sign_X, logdet_X = np.linalg.slogdet(
        XtAinvX
    )

    if sign_X <= 0:
        return np.inf if not return_fit else None

    try:
        chol_X = cho_factor(
            XtAinvX,
            lower=True,
            check_finite=False,
        )

        beta_hat = cho_solve(
            chol_X,
            XtAinvy,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    q = float(
        yAinvy
        - beta_hat @ XtAinvy
    )

    df_reml = m - p_fixed

    if q <= 0 or df_reml <= 0:
        return np.inf if not return_fit else None

    logdet_M = 2.0 * np.sum(
        np.log(np.diag(chol_M[0]))
    )

    logdet_A = (
        m * np.log(delta)
        + logdet_M
    )

    objective = (
        logdet_A
        + logdet_X
        + df_reml * np.log(q / df_reml)
    )

    if return_fit:
        return {
            "objective": float(objective),
            "beta_hat": beta_hat,
            "q": q,
            "delta": delta,
            "df_reml": int(df_reml),
        }

    return float(objective)


def fit_section2_reml_gls(X, y, pathogen_index, K, static=None):
    if static is None:
        static = prepare_reml_static(
            X,
            pathogen_index,
            K,
        )

    working = add_y_to_reml_static(
        static,
        y,
    )

    optimization = minimize_scalar(
        lambda log_delta: evaluate_profile_reml(
            log_delta,
            working,
            return_fit=False,
        ),
        bounds=(-8.0, 8.0),
        method="bounded",
        options={
            "xatol": 1e-4,
            "maxiter": 100,
        },
    )

    if not optimization.success:
        raise RuntimeError(
            "REML optimization failed: "
            + str(optimization.message)
        )

    fit = evaluate_profile_reml(
        optimization.x,
        working,
        return_fit=True,
    )

    if fit is None:
        raise RuntimeError(
            "Final REML/GLS evaluation failed."
        )

    sigma_g2_hat = (
        fit["q"]
        / fit["df_reml"]
    )

    sigma_e2_hat = (
        fit["delta"]
        * sigma_g2_hat
    )

    return {
        "beta_hat": fit["beta_hat"],
        "sigma_g2_hat": float(sigma_g2_hat),
        "sigma_e2_hat": float(sigma_e2_hat),
        "delta_hat": float(fit["delta"]),
        "reml_objective": float(fit["objective"]),
        "optimization_nfev": int(optimization.nfev),
        "static": static,
    }


def unpack_beta(beta_hat):
    start_C = 1
    stop_C = start_C + D_C

    start_P = stop_C
    stop_P = start_P + D_P

    start_B = stop_P

    alpha_hat = float(beta_hat[0])
    beta_C_hat = beta_hat[start_C:stop_C]
    beta_P_hat = beta_hat[start_P:stop_P]
    B_hat = beta_hat[start_B:].reshape(
        D_C,
        D_P,
    )

    return (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    )


print("Cell 4: PASS")


In [ ]:
#@title Cell 5 - Generate and QC one weak/strong association dataset

EXAMPLE_SEED = MASTER_SEED

example = prepare_association_dataset(
    EXAMPLE_SEED
)

weak_anchor_pathogens = (
    example["weak"][
        "anchor_pathogen_indices"
    ]
)

weak_anchor_plasmids = (
    example["weak"][
        "anchor_plasmid_indices"
    ]
)

if not np.array_equal(
    weak_anchor_pathogens,
    example["strong"][
        "anchor_pathogen_indices"
    ],
):
    raise RuntimeError(
        "Weak and strong designs did not use the same anchor pathogens."
    )

if not np.array_equal(
    weak_anchor_plasmids,
    example["strong"][
        "anchor_plasmid_indices"
    ],
):
    raise RuntimeError(
        "Weak and strong designs did not use the same anchor plasmids."
    )

X_anchor = build_minimal_anchor_subdesign(
    example["C"],
    example["P"],
    weak_anchor_pathogens,
    weak_anchor_plasmids,
)

anchor_rank = np.linalg.matrix_rank(
    X_anchor
)

association_qc = pd.DataFrame([
    {
        "design":
            "weak_association",

        "association_strength":
            example["weak"][
                "association_strength"
            ],

        "observed_rows":
            len(
                example["weak"]["y"]
            ),

        "observed_Pplus":
            int(
                example["weak"][
                    "observed_pplus_grid"
                ].sum()
            ),

        "min_pathogens_per_plasmid":
            int(
                example["weak"][
                    "plasmid_counts"
                ].min()
            ),

        "max_pathogens_per_plasmid":
            int(
                example["weak"][
                    "plasmid_counts"
                ].max()
            ),

        "mean_abs_score_centre_distance":
            example["weak"][
                "mean_abs_score_centre_distance"
            ],

        "score_centre_correlation":
            example["weak"][
                "score_centre_correlation"
            ],

        "support_repair_swaps":
            example["weak"][
                "support_repair_swaps"
            ],

        "design_rank":
            np.linalg.matrix_rank(
                example["weak"]["X"]
            ),
    },
    {
        "design":
            "strong_association",

        "association_strength":
            example["strong"][
                "association_strength"
            ],

        "observed_rows":
            len(
                example["strong"]["y"]
            ),

        "observed_Pplus":
            int(
                example["strong"][
                    "observed_pplus_grid"
                ].sum()
            ),

        "min_pathogens_per_plasmid":
            int(
                example["strong"][
                    "plasmid_counts"
                ].min()
            ),

        "max_pathogens_per_plasmid":
            int(
                example["strong"][
                    "plasmid_counts"
                ].max()
            ),

        "mean_abs_score_centre_distance":
            example["strong"][
                "mean_abs_score_centre_distance"
            ],

        "score_centre_correlation":
            example["strong"][
                "score_centre_correlation"
            ],

        "support_repair_swaps":
            example["strong"][
                "support_repair_swaps"
            ],

        "design_rank":
            np.linalg.matrix_rank(
                example["strong"]["X"]
            ),
    },
])

MASK_WEAK_PATH = (
    OUTPUT_DIR
    / "05_example_weak_association_mask_200x20.csv"
)

MASK_STRONG_PATH = (
    OUTPUT_DIR
    / "05_example_strong_association_mask_200x20.csv"
)

pd.DataFrame(
    example["weak"][
        "observed_pplus_grid"
    ].astype(int),
    index=[
        f"C{i}"
        for i in range(
            1,
            N_PATHOGENS + 1,
        )
    ],
    columns=[
        f"P{j}"
        for j in range(
            1,
            N_PLASMIDS + 1,
        )
    ],
).to_csv(
    MASK_WEAK_PATH
)

pd.DataFrame(
    example["strong"][
        "observed_pplus_grid"
    ].astype(int),
    index=[
        f"C{i}"
        for i in range(
            1,
            N_PATHOGENS + 1,
        )
    ],
    columns=[
        f"P{j}"
        for j in range(
            1,
            N_PLASMIDS + 1,
        )
    ],
).to_csv(
    MASK_STRONG_PATH
)

print("=" * 90)
print("CELL 5 — SIMULATION 5 ASSOCIATION-DESIGN QC")
print("=" * 90)

print(
    f"Complete observations before masking: "
    f"{N_PATHOGENS * (N_PLASMIDS + 1):,}"
)

print(
    f"Observed rows per design:             "
    f"{len(example['weak']['y']):,}"
)

print(
    f"Fixed-effect columns:                 "
    f"{example['weak']['X'].shape[1]}"
)

print(
    f"K dimensions:                        "
    f"{example['K'].shape}"
)

print("\nMinimal full-rank anchor factorial:")
print(
    f"Anchor pathogens:                     "
    f"{len(weak_anchor_pathogens)}"
)
print(
    f"Anchor plasmids:                      "
    f"{len(weak_anchor_plasmids)}"
)
print(
    f"Mandatory anchor P+ combinations:     "
    f"{example['weak']['mandatory_anchor_Pplus_count']}"
)
print(
    f"Anchor factorial dimensions:          "
    f"{X_anchor.shape}"
)
print(
    f"Anchor factorial rank:                "
    f"{anchor_rank}"
)
print(
    "Anchor plasmids:                     "
    + ", ".join(
        example["weak"][
            "anchor_plasmid_ids"
        ]
    )
)

print("\nAssociation diagnostics:")
display(
    association_qc
)

if (
    anchor_rank
    != example["X_complete"].shape[1]
):
    raise RuntimeError(
        "The minimal anchor factorial is not full rank."
    )

if (
    example["strong"][
        "score_centre_correlation"
    ]
    <= example["weak"][
        "score_centre_correlation"
    ]
):
    raise RuntimeError(
        "Strong design did not produce a larger association correlation."
    )

if (
    example["strong"][
        "mean_abs_score_centre_distance"
    ]
    >= example["weak"][
        "mean_abs_score_centre_distance"
    ]
):
    raise RuntimeError(
        "Strong design did not reduce chromosome/plasmid preference distance."
    )

print("\nSaved:")
print(
    MASK_WEAK_PATH
)
print(
    MASK_STRONG_PATH
)

print("\nCell 5: PASS")


In [ ]:
#@title Cell 6 - Fit weak- and strong-association designs

fits = {}
statics = {}
fit_rows = []
beta_rows = []

for label in ["weak", "strong"]:
    t0 = time.time()
    static = prepare_reml_static(
        example[label]["X"],
        example[label]["pathogen_index"],
        example["K"],
    )
    fit = fit_section2_reml_gls(
        example[label]["X"],
        example[label]["y"],
        example[label]["pathogen_index"],
        example["K"],
        static=static,
    )
    fits[label] = fit
    statics[label] = static
    alpha_hat, beta_C_hat, beta_P_hat, B_hat = unpack_beta(fit["beta_hat"])
    fit_rows.append({
        "design": f"{label}_association",
        "alpha_hat": alpha_hat,
        "sigma_g2_hat": fit["sigma_g2_hat"],
        "sigma_e2_hat": fit["sigma_e2_hat"],
        "variance_ratio_hat": fit["delta_hat"],
        "fit_time_seconds": time.time() - t0,
    })
    beta_rows.append(beta_P_hat)

fit_summary = pd.DataFrame(fit_rows)
beta_comparison = pd.DataFrame({
    "feature": PLASMID_FEATURE_LABELS,
    "true_beta_P": BETA_P_TRUE,
    "weak_association_fit": beta_rows[0],
    "strong_association_fit": beta_rows[1],
})

print("=" * 90)
print("CELL 6 — WEAK VS STRONG ASSOCIATION FIT")
print("=" * 90)
print(f"True alpha:       {ALPHA_TRUE:.6f}")
print(f"True sigma_g^2:   {SIGMA_G2_TRUE:.6f}")
print(f"True sigma_e^2:   {SIGMA_E2_TRUE:.6f}")
print("\\nFit comparison:")
display(fit_summary)
print("\\nPlasmid main-effect recovery:")
display(beta_comparison)
print("\\nCell 6: PASS")


In [ ]:
#@title Cell 7 - Compare Delta and DeltaDelta recovery under weak and strong association

def true_and_estimated_effects(dataset, fit):
    C = dataset["C"]
    P = dataset["P"]
    alpha_hat, beta_C_hat, beta_P_hat, B_hat = unpack_beta(fit["beta_hat"])

    y0_true = ALPHA_TRUE + C @ dataset["beta_C_true"]
    delta_true = (
        (P @ dataset["beta_P_true"])[None, :]
        + C @ dataset["B_true"] @ P.T
    )
    yij_true = y0_true[:, None] + delta_true

    y0_hat = alpha_hat + C @ beta_C_hat
    delta_hat = (P @ beta_P_hat)[None, :] + C @ B_hat @ P.T
    yij_hat = y0_hat[:, None] + delta_hat

    return {
        "y0_true": y0_true, "y0_hat": y0_hat,
        "yij_true": yij_true, "yij_hat": yij_hat,
        "delta_true": delta_true, "delta_hat": delta_hat,
    }


def basic_metrics(true_values, estimated_values):
    t = np.asarray(true_values, dtype=float).ravel()
    e = np.asarray(estimated_values, dtype=float).ravel()
    error = e - t
    nonzero = np.abs(t) > 1e-12
    sign_accuracy = np.mean(np.sign(e[nonzero]) == np.sign(t[nonzero])) if nonzero.any() else np.nan
    return {
        "bias": float(np.mean(error)),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "sign_accuracy": float(sign_accuracy),
    }


def pairwise_delta_delta(delta_matrix):
    upper_i, upper_k = np.triu_indices(delta_matrix.shape[0], k=1)
    return delta_matrix[upper_i, :] - delta_matrix[upper_k, :], upper_i, upper_k


def evaluate_association_fit(dataset, fit, design_label):
    effects = true_and_estimated_effects(dataset, fit)
    dd_true, pair_i, pair_k = pairwise_delta_delta(effects["delta_true"])
    dd_hat, _, _ = pairwise_delta_delta(effects["delta_hat"])

    withheld = dataset[design_label]["withheld_pplus_grid"]
    observed = dataset[design_label]["observed_pplus_grid"]
    dd_withheld = withheld[pair_i, :] | withheld[pair_k, :]

    groups = {
        "delta_all": (effects["delta_true"], effects["delta_hat"]),
        "delta_withheld": (effects["delta_true"][withheld], effects["delta_hat"][withheld]),
        "delta_observed": (effects["delta_true"][observed], effects["delta_hat"][observed]),
        "delta_delta_all": (dd_true, dd_hat),
        "delta_delta_withheld": (dd_true[dd_withheld], dd_hat[dd_withheld]),
    }

    metrics = {}
    for name, (truth, estimate) in groups.items():
        m = basic_metrics(truth, estimate)
        for key, value in m.items():
            metrics[f"{name}_{key}"] = value

    metrics["sigma_g2_hat"] = fit["sigma_g2_hat"]
    metrics["sigma_e2_hat"] = fit["sigma_e2_hat"]

    return {
        "effects": effects,
        "dd_true": dd_true,
        "dd_hat": dd_hat,
        "pair_i": pair_i,
        "pair_k": pair_k,
        "metrics": metrics,
    }


evaluations = {
    label: evaluate_association_fit(example, fits[label], label)
    for label in ["weak", "strong"]
}

comparison_rows = []
for label in ["weak", "strong"]:
    row = {"design": f"{label}_association"}
    row.update(evaluations[label]["metrics"])
    comparison_rows.append(row)
example_comparison = pd.DataFrame(comparison_rows)

print("=" * 90)
print("CELL 7 — EFFECT RECOVERY UNDER WEAK VS STRONG ASSOCIATION")
print("=" * 90)
display(example_comparison)
print("\\nPrimary Simulation 5 comparison:")
print(f"Delta all-grid RMSE: weak={evaluations['weak']['metrics']['delta_all_rmse']:.6f}, strong={evaluations['strong']['metrics']['delta_all_rmse']:.6f}")
print(f"DeltaDelta all-grid RMSE: weak={evaluations['weak']['metrics']['delta_delta_all_rmse']:.6f}, strong={evaluations['strong']['metrics']['delta_delta_all_rmse']:.6f}")
print(f"Withheld Delta RMSE: weak={evaluations['weak']['metrics']['delta_withheld_rmse']:.6f}, strong={evaluations['strong']['metrics']['delta_withheld_rmse']:.6f}")
print("\\nCell 7: PASS")


In [ ]:
#@title Cell 8 - Run 100 independent Simulation 5 replicates

def run_one_simulation5_replicate(replicate_index):
    seed = MASTER_SEED + 4000 + int(replicate_index)
    dataset = prepare_association_dataset(seed)
    out = {
        "replicate": int(replicate_index + 1),
        "seed": int(seed),
        "weak_association_correlation": dataset["weak"]["score_centre_correlation"],
        "strong_association_correlation": dataset["strong"]["score_centre_correlation"],
        "weak_mean_distance": dataset["weak"]["mean_abs_score_centre_distance"],
        "strong_mean_distance": dataset["strong"]["mean_abs_score_centre_distance"],
    }

    for label in ["weak", "strong"]:
        static = prepare_reml_static(
            dataset[label]["X"], dataset[label]["pathogen_index"], dataset["K"]
        )
        fit = fit_section2_reml_gls(
            dataset[label]["X"], dataset[label]["y"],
            dataset[label]["pathogen_index"], dataset["K"], static=static,
        )
        ev = evaluate_association_fit(dataset, fit, label)
        for metric, value in ev["metrics"].items():
            out[f"{label}_{metric}"] = value
    return out


replicate_rows = []
start_time = time.time()
for r in range(N_SIM_REPLICATES):
    print(
        f"Starting replicate {r + 1:3d}/{N_SIM_REPLICATES}...",
        flush=True,
    )

    replicate_start = time.time()

    replicate_rows.append(
        run_one_simulation5_replicate(r)
    )

    print(
        f"Finished replicate {r + 1:3d}/{N_SIM_REPLICATES} "
        f"in {time.time() - replicate_start:.1f} seconds",
        flush=True,
    )

    if (r + 1) % 10 == 0 or r == 0 or r + 1 == N_SIM_REPLICATES:
        print(f"Completed {r + 1:3d}/{N_SIM_REPLICATES} replicates ({time.time() - start_time:.1f} seconds elapsed)")

replicate_results = pd.DataFrame(replicate_rows)
REPLICATE_RESULTS_PATH = OUTPUT_DIR / "05_scenario5_association_replicate_metrics.csv"
replicate_results.to_csv(REPLICATE_RESULTS_PATH, index=False)
print("\\nSaved:")
print(REPLICATE_RESULTS_PATH)
print("\\nCell 8: PASS")


In [ ]:
#@title Cell 9 - Summarize the 100-replicate Simulation 5 benchmark

PRIMARY_METRICS = [
    "weak_delta_all_bias", "weak_delta_all_rmse", "weak_delta_all_sign_accuracy",
    "strong_delta_all_bias", "strong_delta_all_rmse", "strong_delta_all_sign_accuracy",
    "weak_delta_delta_all_bias", "weak_delta_delta_all_rmse", "weak_delta_delta_all_sign_accuracy",
    "strong_delta_delta_all_bias", "strong_delta_delta_all_rmse", "strong_delta_delta_all_sign_accuracy",
    "weak_delta_withheld_rmse", "strong_delta_withheld_rmse",
    "weak_delta_delta_withheld_rmse", "strong_delta_delta_withheld_rmse",
]
SUPPORTING_METRICS = [
    "weak_association_correlation", "strong_association_correlation",
    "weak_mean_distance", "strong_mean_distance",
    "weak_sigma_g2_hat", "strong_sigma_g2_hat",
    "weak_sigma_e2_hat", "strong_sigma_e2_hat",
]

summary_rows = []
for metric in PRIMARY_METRICS + SUPPORTING_METRICS:
    values = replicate_results[metric].to_numpy(dtype=float)
    summary_rows.append({
        "metric": metric,
        "mean": float(np.nanmean(values)),
        "sd": float(np.nanstd(values, ddof=1)),
        "median": float(np.nanmedian(values)),
        "q025": float(np.nanquantile(values, 0.025)),
        "q975": float(np.nanquantile(values, 0.975)),
    })
scenario5_summary = pd.DataFrame(summary_rows)
SUMMARY_PATH = OUTPUT_DIR / "05_scenario5_chromosome_plasmid_association_summary.csv"
scenario5_summary.to_csv(SUMMARY_PATH, index=False)

print("=" * 90)
print("SCENARIO 5 — 100-REPLICATE SUMMARY")
print("=" * 90)
print("\\nPrimary comparison:")
display(scenario5_summary[scenario5_summary["metric"].isin(PRIMARY_METRICS)].reset_index(drop=True))
print("\\nAssociation and variance diagnostics:")
display(scenario5_summary[scenario5_summary["metric"].isin(SUPPORTING_METRICS)].reset_index(drop=True))

print("\\nDirect loss from stronger chromosome–plasmid association:")
print(f"Mean Delta RMSE ratio, strong/weak:      {(replicate_results['strong_delta_all_rmse'] / replicate_results['weak_delta_all_rmse']).mean():.3f}")
print(f"Mean DeltaDelta RMSE ratio, strong/weak: {(replicate_results['strong_delta_delta_all_rmse'] / replicate_results['weak_delta_delta_all_rmse']).mean():.3f}")
print("\\nSaved:")
print(SUMMARY_PATH)
print("\\nCell 9: PASS")


In [ ]:
#@title Cell 10 - Representative parametric bootstrap under weak and strong association

def simulate_parametric_bootstrap_y(rng, X, pathogen_index, K, beta_hat, sigma_g2_hat, sigma_e2_hat):
    u_star = draw_correlated_host_effect(rng, K, sigma_g2_hat)
    epsilon_star = rng.normal(0.0, np.sqrt(sigma_e2_hat), size=X.shape[0])
    return X @ beta_hat + u_star[pathogen_index] + epsilon_star


def effects_from_beta(C, P, beta_hat):
    alpha_hat, beta_C_hat, beta_P_hat, B_hat = unpack_beta(beta_hat)
    return (P @ beta_P_hat)[None, :] + C @ B_hat @ P.T


def interval_summary(bootstrap_values, truth_values, label):
    truth = np.asarray(truth_values, dtype=float).ravel()
    low = np.quantile(bootstrap_values, 0.025, axis=0)
    high = np.quantile(bootstrap_values, 0.975, axis=0)
    return {
        "effect": label,
        "number_of_effects": int(len(truth)),
        "fraction_CI_excludes_zero": float(np.mean((low > 0) | (high < 0))),
        "fraction_CI_contains_known_truth": float(np.mean((low <= truth) & (truth <= high))),
        "mean_CI_width": float(np.mean(high - low)),
    }


delta_true = evaluations["weak"]["effects"]["delta_true"]
dd_true = evaluations["weak"]["dd_true"]
bootstrap_rng = np.random.default_rng(MASTER_SEED + 900000)

store = {}
for label in ["weak", "strong"]:
    store[label] = {
        "delta": np.empty((N_BOOTSTRAP, N_PATHOGENS * N_PLASMIDS), dtype=np.float32),
        "dd": np.empty((N_BOOTSTRAP, dd_true.size), dtype=np.float32),
    }

start_time = time.time()
for b in range(N_BOOTSTRAP):
    for label in ["weak", "strong"]:
        fit = fits[label]
        y_star = simulate_parametric_bootstrap_y(
            bootstrap_rng, example[label]["X"], example[label]["pathogen_index"],
            example["K"], fit["beta_hat"], fit["sigma_g2_hat"], fit["sigma_e2_hat"],
        )
        fit_star = fit_section2_reml_gls(
            example[label]["X"], y_star, example[label]["pathogen_index"],
            example["K"], static=statics[label],
        )
        delta_star = effects_from_beta(example["C"], example["P"], fit_star["beta_hat"])
        dd_star, _, _ = pairwise_delta_delta(delta_star)
        store[label]["delta"][b, :] = delta_star.ravel().astype(np.float32)
        store[label]["dd"][b, :] = dd_star.ravel().astype(np.float32)

    if (b + 1) % 20 == 0 or b == 0 or b + 1 == N_BOOTSTRAP:
        print(f"Completed {b + 1:3d}/{N_BOOTSTRAP} paired bootstrap refits ({time.time() - start_time:.1f} seconds elapsed)")

bootstrap_summary = pd.DataFrame([
    interval_summary(store["weak"]["delta"], delta_true, "Delta_weak_association"),
    interval_summary(store["strong"]["delta"], delta_true, "Delta_strong_association"),
    interval_summary(store["weak"]["dd"], dd_true, "DeltaDelta_weak_association"),
    interval_summary(store["strong"]["dd"], dd_true, "DeltaDelta_strong_association"),
])
BOOTSTRAP_SUMMARY_PATH = OUTPUT_DIR / "05_scenario5_representative_bootstrap_summary.csv"
bootstrap_summary.to_csv(BOOTSTRAP_SUMMARY_PATH, index=False)
display(bootstrap_summary)
print("\\nImportant:")
print("Weak and strong fits use the same complete simulated truth and the same number of observed states; only the chromosome–plasmid association pattern differs.")
print("\\nSaved:")
print(BOOTSTRAP_SUMMARY_PATH)
print("\\nCell 10: PASS")


In [ ]:
#@title Cell 11 - Optional repeated-dataset bootstrap coverage

print("Formal repeated-dataset bootstrap coverage remains optional and is disabled by default, as in Simulations 1-4.")
if RUN_FULL_BOOTSTRAP_COVERAGE:
    print("RUN_FULL_BOOTSTRAP_COVERAGE=True was requested, but this notebook does not automatically launch the very expensive repeated-dataset bootstrap calculation. Inspect Cell 10 first.")
else:
    print("RUN_FULL_BOOTSTRAP_COVERAGE=False: no formal repeated-dataset coverage run was performed.")
print("\\nCell 11: PASS")


In [ ]:
#@title Cell 12 - Final QC and output manifest

required_output_files = [
    MASK_WEAK_PATH,
    MASK_STRONG_PATH,
    REPLICATE_RESULTS_PATH,
    SUMMARY_PATH,
    BOOTSTRAP_SUMMARY_PATH,
]
missing_outputs = [str(path) for path in required_output_files if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("Required Simulation 5 output(s) are missing:\\n" + "\\n".join(missing_outputs))

manifest = {
    "notebook": "11_Simulation_05_Chromosome_Plasmid_Association.ipynb",
    "scenario": "Simulation 05 - chromosome-plasmid association",
    "central_question": "Does restricted chromosome-plasmid mixing reduce recovery when observation count is held constant?",
    "simulation1_biology_unchanged": True,
    "scenario5_change": {
        "weak_association_strength": WEAK_ASSOCIATION_STRENGTH,
        "strong_association_strength": STRONG_ASSOCIATION_STRENGTH,
        "observed_plasmids_per_pathogen": N_OBSERVED_PLASMIDS_PER_PATHOGEN,
        "P0_retained_for_all_pathogens": True,
        "same_observation_count_weak_and_strong": True,
        "association_axis": "PC1 of the 60-dimensional chromosome feature matrix",
    },
    "simulation_settings": {
        "pathogens": N_PATHOGENS,
        "plasmids": N_PLASMIDS,
        "observations_per_design": N_PATHOGENS * (N_OBSERVED_PLASMIDS_PER_PATHOGEN + 1),
        "alpha": ALPHA_TRUE,
        "sigma_g": SIGMA_G_TRUE,
        "sigma_e": SIGMA_E_TRUE,
        "simulation_replicates": N_SIM_REPLICATES,
        "bootstrap_replicates": N_BOOTSTRAP,
    },
    "outputs": [str(path) for path in required_output_files],
}
MANIFEST_PATH = OUTPUT_DIR / "05_scenario5_manifest.json"
with open(MANIFEST_PATH, "w") as handle:
    json.dump(manifest, handle, indent=2)

print("=" * 90)
print("SIMULATION 5 NOTEBOOK COMPLETE")
print("=" * 90)
print(f"Manifest: {MANIFEST_PATH}")
print(f"Required output files checked: {len(required_output_files)}")
print("\\nOnly the chromosome–plasmid observation pattern differs between the weak- and strong-association fits.")
print("Cell 12: PASS")
